# Job ETL

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. Ele organiza e prepara os dados para que possam ser usados em análises ou relatórios. No caso desse projeto, a fonte será do arquivo Complete_Pokedex_V1.1.csv e o resultado será utilizado na camada gold.

### Job ETL: Extração, Transformação e Carregamento


Aqui executamos o processo de transformação (ETL). Esta célula utiliza o `dataFrame` carregado na etapa anterior, aplica as regras de limpeza e, por fim, salva o resultado em um novo arquivo CSV 

In [1]:
import pandas as pd

#=========EXTRAÇÃO==========

dataFrame = pd.read_csv('../Data_Layer/raw/Complete_Pokedex_V1.1.csv')

#=========TRANSFORMAÇÃO==========

# Apaga colunas
colunas_para_apagar = [
'ability_1',
'ability_2',
'ability_3',
'number_pokemon_with_typing',
'primary_color'
]

dataFrame = dataFrame.drop(columns=colunas_para_apagar)
dataFrame = dataFrame.drop(columns=["mean", "standard_deviation", "exp_to_level_100", "can_evolve", "final_evolution", "is_default", "baby_pokemon", "genus", "egg_group_1", "egg_group_2", "shape", "bmi", "special_attack", "special_defense", "speed"])

# Remove tuplas duplicadas do Pokedex_number 
dataFrame = dataFrame.drop_duplicates(subset=['pokedex_number'])

# Trata nulos, transforma para Indefinido
dataFrame[["type_2", "evolves_from"]] = dataFrame[["type_2", "evolves_from"]].fillna("Indefinido")



#=========LOAD==========

# salva em novo csv sem o indice automatico
dataFrame.to_csv("Complete_Pokedex-Tratada.csv", index=False)

print(dataFrame.head())

print("\n Job ETL concluído!")

   pokedex_number   pokemon_name type_1      type_2  height  weight  \
0               1      Bulbasaur  Grass      Poison     0.7     6.9   
1               2        Ivysaur  Grass      Poison     1.0    13.0   
2               3  Mega Venusaur  Grass      Poison     2.4   155.5   
5               4     Charmander   Fire  Indefinido     0.6     8.5   
6               5     Charmeleon   Fire  Indefinido     1.1    19.0   

   hit_points  attack  defense  total_stats  ...  against_ground  \
0          45      49       49          318  ...             1.0   
1          60      62       63          405  ...             1.0   
2          80     100      123          625  ...             1.0   
5          39      52       43          309  ...             2.0   
6          58      64       58          405  ...             2.0   

   against_flying  against_psychic  against_bug against_rock against_ghost  \
0             2.0              2.0          1.0          1.0           1.0   
1       

#### LOAD: Populando o Banco PostgreQSL

é realizada a conexão com o banco para criar e popular a tabela de acordo com os dados do arquivo "Complete_Pokedex-Tratada.csv" .

In [2]:
import pandas as pd
import psycopg2
import time

dataFrame = pd.read_csv("Complete_Pokedex-Tratada.csv")

# Conecta no PostgreSQL
while True:
    try:
        conexao = psycopg2.connect(
            host="postgres", # localhost
            port=5432,
            database="silver_db",
            user="silver_user",
            password="silver_password"
        )
        break
    except psycopg2.OperationalError:
        print("O banco não está pronto, aguardando 3 segundos...")
        time.sleep(3)

# cria o cursor
cursor = conexao.cursor()

# com o cursos cria a tabela 

cursor.execute("""
CREATE TABLE IF NOT EXISTS pokemon (
    pokedex_number INT NOT NULL PRIMARY KEY,
    pokemon_name VARCHAR(50) NOT NULL,
    type_1 VARCHAR(50) NOT NULL,
    type_2 VARCHAR(50),
    height DOUBLE PRECISION NOT NULL,
    weight DOUBLE PRECISION NOT NULL,
    hit_points INT NOT NULL,
    attack INT NOT NULL,
    defense INT NOT NULL,
    total_stats INT NOT NULL,
    capture_rate INT NOT NULL,
    generation INT NOT NULL,
    base_happiness INT NOT NULL,
    base_experience INT NOT NULL,
    exp_type VARCHAR(50) NOT NULL,
    evolves_from VARCHAR(50),
    mega_evolution BOOLEAN NOT NULL,
    alolan_form BOOLEAN NOT NULL,
    galarian_form BOOLEAN NOT NULL,
    forms_switchable BOOLEAN NOT NULL,
    legendary BOOLEAN NOT NULL,
    mythical BOOLEAN NOT NULL,
    genderless BOOLEAN NOT NULL,
    female_rate DOUBLE PRECISION NOT NULL,
    egg_cycles INT NOT NULL,
    against_normal DOUBLE PRECISION NOT NULL,
    against_fire DOUBLE PRECISION NOT NULL,
    against_water DOUBLE PRECISION NOT NULL,
    against_electric DOUBLE PRECISION NOT NULL,
    against_grass DOUBLE PRECISION NOT NULL,
    against_ice DOUBLE PRECISION NOT NULL,
    against_fighting DOUBLE PRECISION NOT NULL,
    against_poison DOUBLE PRECISION NOT NULL,
    against_ground DOUBLE PRECISION NOT NULL,
    against_flying DOUBLE PRECISION NOT NULL,
    against_psychic DOUBLE PRECISION NOT NULL,
    against_bug DOUBLE PRECISION NOT NULL,
    against_rock DOUBLE PRECISION NOT NULL,
    against_ghost DOUBLE PRECISION NOT NULL,
    against_dragon DOUBLE PRECISION NOT NULL,
    against_dark DOUBLE PRECISION NOT NULL,
    against_steel DOUBLE PRECISION NOT NULL,
    against_fairy DOUBLE PRECISION NOT NULL
)
""")

conexao.commit()

# Inserir dados do CSV
with open('Complete_Pokedex-Tratada.csv', 'r') as arqCSV:
    next(arqCSV)  
    cursor.copy_from(arqCSV, 'pokemon', sep=',')

conexao.commit()

# Fechando conexão
cursor.close()
conexao.close()

print("concluido!")

concluido!


# PARTE DO PROFESSOR

In [3]:
import pandas as pd

df = pd.read_csv('../Data_Layer/raw/Complete_Pokedex_V1.1.csv')


In [4]:
df.head()

,pokedex_number,pokemon_name,type_1,type_2,ability_1,ability_2,ability_3,number_pokemon_with_typing,primary_color,shape,...,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,1,Bulbasaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,2,Ivysaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,3,Mega Venusaur,Grass,Poison,Thick Fat,NaN,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,3,Venusaur,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
4,3,Venusaur Gmax,Grass,Poison,Overgrow,Chlorophyll,NaN,15,Green,Quadruped,...,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5


In [5]:
df.columns

Index(['pokedex_number', 'pokemon_name', 'type_1', 'type_2', 'ability_1',
       'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color',
       'shape', 'height', 'weight', 'bmi', 'hit_points', 'attack', 'defense',
       'special_attack', 'special_defense', 'speed', 'total_stats', 'mean',
       'standard_deviation', 'capture_rate', 'generation', 'base_happiness',
       'base_experience', 'exp_type', 'exp_to_level_100', 'can_evolve',
       'evolves_from', 'final_evolution', 'mega_evolution', 'is_default',
       'baby_pokemon', 'alolan_form', 'galarian_form', 'forms_switchable',
       'legendary', 'mythical', 'genderless', 'female_rate', 'genus',
       'egg_group_1', 'egg_group_2', 'egg_cycles', 'against_normal',
       'against_fire', 'against_water', 'against_electric', 'against_grass',
       'against_ice', 'against_fighting', 'against_poison', 'against_ground',
       'against_flying', 'against_psychic', 'against_bug', 'against_rock',
       'against_ghost', '

In [8]:
df['type_1'].unique()

array(['Grass', 'Fire', 'Water', 'Bug', 'Normal', 'Dark', 'Poison',
       'Electric', 'Ice', 'Ground', 'Fairy', 'Steel', 'Fighting',
       'Psychic', 'Rock', 'Ghost', 'Dragon', 'Flying'], dtype=object)

In [11]:
df.columns

Index(['pokedex_number', 'pokemon_name', 'type_1', 'type_2', 'ability_1',
       'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color',
       'shape', 'height', 'weight', 'bmi', 'hit_points', 'attack', 'defense',
       'special_attack', 'special_defense', 'speed', 'total_stats', 'mean',
       'standard_deviation', 'capture_rate', 'generation', 'base_happiness',
       'base_experience', 'exp_type', 'exp_to_level_100', 'can_evolve',
       'evolves_from', 'final_evolution', 'mega_evolution', 'is_default',
       'baby_pokemon', 'alolan_form', 'galarian_form', 'forms_switchable',
       'legendary', 'mythical', 'genderless', 'female_rate', 'genus',
       'egg_group_1', 'egg_group_2', 'egg_cycles', 'against_normal',
       'against_fire', 'against_water', 'against_electric', 'against_grass',
       'against_ice', 'against_fighting', 'against_poison', 'against_ground',
       'against_flying', 'against_psychic', 'against_bug', 'against_rock',
       'against_ghost', '

In [ ]:
df_temp = df[df['type_1'] == 'Grass']

,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,against_poison,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,1.0,2.0,0.50,0.50,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,1.0,2.0,0.50,0.50,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,1.0,2.0,0.50,0.50,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,1.0,2.0,0.50,0.50,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
4,1.0,2.0,0.50,0.50,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1037,1.0,1.0,0.25,0.25,0.25,4.0,1.0,2.0,0.5,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0
1038,1.0,1.0,0.25,0.25,0.25,4.0,1.0,2.0,0.5,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0
1039,1.0,1.0,0.25,0.25,0.25,4.0,1.0,2.0,0.5,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0
1040,1.0,1.0,0.25,0.25,0.25,4.0,1.0,2.0,0.5,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0


In [34]:
df_tipo = df.drop(columns = ['pokedex_number', 'pokemon_name', 'ability_1',
       'ability_2', 'ability_3', 'number_pokemon_with_typing', 'primary_color',
       'shape', 'height', 'weight', 'bmi', 'hit_points', 'attack', 'defense',
       'special_attack', 'special_defense', 'speed', 'total_stats', 'mean',
       'standard_deviation', 'capture_rate', 'generation', 'base_happiness',
       'base_experience', 'exp_type', 'exp_to_level_100', 'can_evolve',
       'evolves_from', 'final_evolution', 'mega_evolution', 'is_default',
       'baby_pokemon', 'alolan_form', 'galarian_form', 'forms_switchable',
       'legendary', 'mythical', 'genderless', 'female_rate', 'genus',
       'egg_group_1', 'egg_group_2', 'egg_cycles',])

In [35]:
df_tipo

,type_1,type_2,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,against_poison,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
1,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
2,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
3,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
4,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113,Ice,NaN,1.0,2.0,1.0,1.0,1.00,0.5,2.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0
1114,Ghost,NaN,0.0,1.0,1.0,1.0,1.00,1.0,0.0,0.5,1.0,1.0,1.0,0.5,1.0,2.0,1.0,2.0,1.0,1.0
1115,Psychic,Grass,1.0,2.0,0.5,0.5,0.50,2.0,0.5,2.0,0.5,2.0,0.5,4.0,1.0,2.0,1.0,2.0,1.0,1.0
1116,Psychic,Ice,1.0,2.0,1.0,1.0,1.00,0.5,1.0,1.0,1.0,1.0,0.5,2.0,2.0,2.0,1.0,2.0,2.0,1.0


In [36]:
df_tipo = df_tipo.fillna('')

In [37]:
df_tipo['Tipo'] = df_tipo['type_1'] + ' ' + df_tipo['type_2']

In [38]:
df_tipo

,type_1,type_2,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,against_poison,...,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy,Tipo
0,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
1,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
2,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
3,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
4,Grass,Poison,1.0,2.0,0.5,0.5,0.25,2.0,0.5,1.0,...,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,Grass Poison
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113,Ice,,1.0,2.0,1.0,1.0,1.00,0.5,2.0,1.0,...,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,Ice
1114,Ghost,,0.0,1.0,1.0,1.0,1.00,1.0,0.0,0.5,...,1.0,1.0,0.5,1.0,2.0,1.0,2.0,1.0,1.0,Ghost
1115,Psychic,Grass,1.0,2.0,0.5,0.5,0.50,2.0,0.5,2.0,...,2.0,0.5,4.0,1.0,2.0,1.0,2.0,1.0,1.0,Psychic Grass
1116,Psychic,Ice,1.0,2.0,1.0,1.0,1.00,0.5,1.0,1.0,...,1.0,0.5,2.0,2.0,2.0,1.0,2.0,2.0,1.0,Psychic Ice


In [39]:
df_tipo = df_tipo.drop_duplicates()

In [40]:
df_tipo.columns

Index(['type_1', 'type_2', 'against_normal', 'against_fire', 'against_water',
       'against_electric', 'against_grass', 'against_ice', 'against_fighting',
       'against_poison', 'against_ground', 'against_flying', 'against_psychic',
       'against_bug', 'against_rock', 'against_ghost', 'against_dragon',
       'against_dark', 'against_steel', 'against_fairy', 'Tipo'],
      dtype='object')

In [41]:
df_tipo = df_tipo[['Tipo','type_1', 'type_2', 'against_normal', 'against_fire', 'against_water',
       'against_electric', 'against_grass', 'against_ice', 'against_fighting',
       'against_poison', 'against_ground', 'against_flying', 'against_psychic',
       'against_bug', 'against_rock', 'against_ghost', 'against_dragon',
       'against_dark', 'against_steel', 'against_fairy']]

In [42]:
df_tipo.reset_index(inplace = True,drop=True)

In [43]:
df_tipo.head()

,Tipo,type_1,type_2,against_normal,against_fire,against_water,against_electric,against_grass,against_ice,against_fighting,...,against_ground,against_flying,against_psychic,against_bug,against_rock,against_ghost,against_dragon,against_dark,against_steel,against_fairy
0,Grass Poison,Grass,Poison,1.0,2.00,0.5,0.5,0.25,2.0,0.5,...,1.0,2.0,2.0,1.00,1.0,1.0,1.0,1.0,1.0,0.5
1,Fire,Fire,,1.0,0.50,2.0,1.0,0.50,0.5,1.0,...,2.0,1.0,1.0,0.50,2.0,1.0,1.0,1.0,0.5,0.5
2,Fire Flying,Fire,Flying,1.0,0.50,2.0,2.0,0.25,1.0,0.5,...,0.0,1.0,1.0,0.25,4.0,1.0,1.0,1.0,0.5,0.5
3,Fire Dragon,Fire,Dragon,1.0,0.25,1.0,0.5,0.25,1.0,1.0,...,2.0,1.0,1.0,0.50,2.0,1.0,2.0,1.0,0.5,1.0
4,Water,Water,,1.0,0.50,0.5,2.0,2.00,0.5,1.0,...,1.0,1.0,1.0,1.00,1.0,1.0,1.0,1.0,0.5,1.0
